-RIDGE REGRESSION
-

In [28]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
X,y=make_regression(n_samples=1000,
                    n_features=1,
                    n_informative=1,
                    n_targets=1,
                    noise=50)
df=pd.DataFrame(X,columns=[f"Feature_{i+1}" for i in range(X.shape[1])])
df['targets']=y

X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['targets'],axis=1), 
                                               df['targets'], 
                                               test_size=0.3, 
                                               random_state=41)
class Ridge:
    def __init__(self,alpha):
        self.alpha=alpha
        self.m=None
        self.b=None
        
    def fit(self,X_train,y_train):
        numerator=0
        denominator=0

        X_train=X_train.values.ravel()
        y_train=y_train.values.ravel()
        
        for i in range(X_train.shape[0]):
            
            numerator+=(y_train[i]-y_train.mean())*(X_train[i]-X_train.mean())
            denominator+=(X_train[i]-X_train.mean())**2
            self.m=numerator/(denominator+self.alpha)
            
            self.b=y_train.mean()-self.m*X_train.mean()
        print(f"Coef_: {self.m} ,Intercept_: {self.b}")
        
    def predict(self,X_test):
        X_test=X_test.values.ravel()
        return self.m*X_test+self.b

ridge=Ridge(alpha=0)
ridge.fit(X_train,y_train)
y_pred=ridge.predict(X_test)
print(f"mse: {mean_squared_error(y_pred, y_test)}, r2_score: {r2_score(y_pred, y_test)}")

from sklearn.linear_model import Ridge
r=Ridge(20)
r.fit(X_train,y_train)
y_pred2=r.predict(X_test)
print(f"Ridge_Coeff: {r.coef_}, Ridge_intercept: {r.intercept_}, R2_Score: {r2_score(y_test, y_pred)}")

Coef_: 44.76171601513049 ,Intercept_: 0.993517878683309
mse: 2438.7088411965465, r2_score: -0.12032629147500806
Ridge_Coeff: [43.53263562], Ridge_intercept: 0.9992039692173825, R2_Score: 0.4328236024238298


In [1]:
#Ridge regression - N Dimensionals

import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

diab = load_diabetes(return_X_y=False, as_frame=True)

df = pd.concat([diab.data, diab.target], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['target']),
    df['target'],
    test_size=0.3,
    random_state=42
)

class Own_MultiRidge:

    def __init__(self, alpha):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):

        X_train = np.insert(X_train, 0, 1, axis=1)

        I = np.identity(X_train.shape[1])
        I[0,0] = 0

        result = np.linalg.inv(
            np.dot(X_train.T, X_train) + self.alpha * I
        ).dot(X_train.T).dot(y_train)

        self.intercept_ = result[0]
        self.coef_ = result[1:]

    def predict(self, X_test):

        return np.dot(X_test, self.coef_) + self.intercept_


mr = Own_MultiRidge(alpha=1.0)

mr.fit(X_train, y_train)

y_pred = mr.predict(X_test)

print(f"Own Ridge R2 Score: {r2_score(y_test, y_pred)}")

print("Own Ridge Coefficients:\n", mr.coef_)
print("Own Ridge Intercept:\n", mr.intercept_)

print("\n<<------------------------------------------------------->>\n")

from sklearn.linear_model import Ridge

r = Ridge(alpha=1.0)

r.fit(X_train, y_train)

r_pred = r.predict(X_test)

print(f"Sklearn Ridge R2 Score: {r2_score(y_test, r_pred)}")

print("Sklearn Ridge Coefficients:\n", r.coef_)
print("Sklearn Ridge Intercept:\n", r.intercept_)

Own Ridge R2 Score: 0.4233440269603016
Own Ridge Coefficients:
 [  45.05421022  -71.94739737  280.71625182  195.21266175   -2.22930269
  -17.54079744 -148.68886188  120.46723979  198.61440137  106.93469215]
Own Ridge Intercept:
 151.86746422977902

<<------------------------------------------------------->>

Sklearn Ridge R2 Score: 0.4233440269603015
Sklearn Ridge Coefficients:
 [  45.05421022  -71.94739737  280.71625182  195.21266175   -2.22930269
  -17.54079744 -148.68886188  120.46723979  198.61440137  106.93469215]
Sklearn Ridge Intercept:
 151.86746422977902


In [2]:
#Ridge Regression - Gradient Descent

import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

class Ridge_GD:

    def __init__(self, alpha, epochs, learning_rate):
        self.alpha = alpha
        self.epochs = epochs
        self.lr = learning_rate
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        m, n = X_train.shape

        theta = np.zeros(n + 1)
        X_train = np.insert(X_train, 0, 1, axis=1)

        for i in range(self.epochs):

            y_hat = np.dot(X_train, theta)
            error = y_hat - y_train
            reg_term = self.alpha * theta
            reg_term[0] = 0
            theta_der = (1/m) * np.dot(X_train.T, error) + reg_term
            theta = theta - self.lr * theta_der

        self.intercept_ = theta[0]
        self.coef_ = theta[1:]

    def predict(self, X_test):

        return np.dot(X_test, self.coef_) + self.intercept_

diab = load_diabetes(return_X_y=False, as_frame=True)
X = diab.data
y = diab.target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Own Ridge GD
own_ridge = Ridge_GD(
    alpha=0.001,
    epochs=5000,
    learning_rate=0.01
)
own_ridge.fit(X_train_scaled, y_train)
own_pred = own_ridge.predict(X_test_scaled)

print("Own Coefficients:\n", own_ridge.coef_)
print("Own Intercept:\n", own_ridge.intercept_)
print(f"\nOwn R2 Score: {r2_score(y_test, own_pred)}")

print('\n<<------------------------------------------------------->>\n')
# Sklearn SGD Ridge
reg = SGDRegressor(
    penalty='l2',
    max_iter=5000,
    eta0=0.01,
    learning_rate='constant',
    alpha=0.001,
    random_state=42
)
reg.fit(X_train_scaled, y_train)
y_pred = reg.predict(X_test_scaled)

print("Sklearn Coefficients:\n", reg.coef_)
print("Sklearn Intercept:\n", reg.intercept_)
print(f"\nSklearn R2 Score: {r2_score(y_test, y_pred)}")

Own Coefficients:
 [  1.57657914 -12.30727787  26.74858217  18.3605872  -18.12356424
   4.65574392  -5.30782406  10.39708985  21.79123179   2.26549412]
Own Intercept:
 153.90291262135784

Own R2 Score: 0.4765574751415407

<<------------------------------------------------------->>

Sklearn Coefficients:
 [  4.60622405  -9.5478166   20.24122225  18.63954987 -14.04296008
   3.2078843   -6.69863557  10.58255324  19.73436533   3.77272582]
Sklearn Intercept:
 [152.84697036]

Sklearn R2 Score: 0.46779542257451834


-LASSO REGRESSION
-

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

diab=load_diabetes(return_X_y=False, as_frame=True)
X=diab.data
y=diab.target
df=pd.concat([X,y],axis=1)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

class Lasso:
    def __init__(self,alpha,epochs,learning_rate):
        self.lr=learning_rate
        self.epochs=epochs
        self.alpha=alpha
        self.coef_=None
        self.intercept_=None

    def fit(self,X_train,y_train):

        m=X_train.shape[0]

        self.intercept_=0
        self.coef_=np.zeros(X_train.shape[1])

        X_train=np.insert(X_train,0,1,axis=1)
        theta=np.insert(self.coef_,0,self.intercept_)

        for i in range(self.epochs):

            y_hat=np.dot(X_train,theta)

            theta_der=(1/m)*np.dot(X_train.T,(y_hat-y_train))

            reg_term=self.alpha*np.sign(theta)
            reg_term[0]=0

            theta_der=theta_der+reg_term

            theta=theta-self.lr*theta_der

        print(theta)

        self.intercept_=theta[0]
        self.coef_=theta[1:]

    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_

la=Lasso(alpha=0.001,epochs=5000,learning_rate=0.01)

la.fit(X_train_scaled,y_train)

la_pred=la.predict(X_test_scaled)

print(r2_score(y_test,la_pred))

print('\n<<------------------------------------------------------->>\n')

reg=SGDRegressor(
    penalty='l1',
    alpha=0.001,
    max_iter=5000,
    eta0=0.01,
    learning_rate='constant',
    random_state=42
)

reg.fit(X_train_scaled,y_train)

y_pred=reg.predict(X_test_scaled)

print(reg.coef_,reg.intercept_)

print(r2_score(y_test,y_pred))

[153.90291262   1.57142828 -12.32428779  26.76424812  18.37865488
 -18.44752794   4.88777438  -5.1654966   10.45878083  21.92203722
   2.24875923]
0.4765147703239577

<<------------------------------------------------------->>

[  4.59009854  -9.56840279  20.2641066   18.66331135 -14.20663457
   3.29499158  -6.62597211  10.62362317  19.80227801   3.75805288] [152.85303606]
0.46784660480961027
